In [487]:
from DIC_LLM.agent_module import ask_agent_with_search
from dotenv import load_dotenv
load_dotenv() # hidden environment variables ignored by git
import logging
# Ignore Warnings of LLM-Outputs
class _NoNonTextWarning(logging.Filter):
    def filter(self, record: logging.LogRecord) -> bool:
        message = record.getMessage()
        # If the log message contains our target string, drop it (return False)
        if "there are non-text parts in the response" in message:
            return False
        return True
logging.getLogger("google_genai.types").addFilter(_NoNonTextWarning())

In [ ]:
# MAIN START 
#Overall Instruction
Team1 = "Team1" #EDIT HERE
Team2 = "Tema2" #EDIT HERE
intro = f"Within the next hours we have the fifa worldcup game between {Team1} vs {Team2}."

#Define Bookmaker Baseline
bookmaker_odds = {
    "Source": "Consensus oddsportal.com",
    "Top_1": f"{Team1} 1-0 {Team2} (6.5)", #EDIT HERE
    "Top_2": f"{Team1} 1-1 {Team2} (7.5)", #EDIT HERE
    "Top_3": f"{Team1} 2-0 {Team2} (8.0)" #EDIT HERE
}


In [ ]:
#Agent A
prompt_agent_a= intro + " "+f'''
You are a quantitative football data scientist analyzing this game in order to predict the Top-3 probabilities of exact outcomes. Base your predictions only on regular time (+ stoppage time), excluding extra time and penalty shootouts.
CRITICAL RULE (Zero Data Leakage): You must absolutely ignore all betting odds, implied probabilities, and bookmaker stats. When using your web search tool, restrict your queries to raw statistical databases (e.g., FBref, Transfermarkt, official FIFA stats). NEVER search for "predictions", "odds", or "betting tips". 
Chain-of-Thought Guardrail: Football is more or less a low-scoring game well approximated by independent Poisson distributions. 
Before outputting any probabilities, mathematically reason step-by-step about the base expected goals (lambda_Team1 and lambda_Team2) for this specific match. 
Variables & Weighting: Base your lambda estimation on these factors: Heavily weight recent match results (last 5-10 games) and short-term xG trends. 
Further factor in: overall attack/defensive strength, historical head-to-head, squad market value, Elo rating, days of rest, and travel distance. 
Neutral Ground Rule: Treat this match as being played on neutral ground with NO home advantage, unless Team 1 or Team 2 is a host nation (USA, Mexico, or Canada). 
Your Output should be strictly & only JSON: Return strictly as a JSON object with: "Poisson_Reasoning": Step-by-step mathematical logic calculating the base lambda_Team1 and lambda_Team2 based 
on the assigned quantitative metrics, explicitly noting recent form and neutral ground dynamics (max 5 sentences). 
"Top_1", "Top_2", "Top_3": The predicted exact scores and their probabilities.
Your Output must be strictly and ONLY a valid JSON object matching the exact structure and keys below. Do not format the Top 3 predictions as flat strings; you must use nested objects with "Score" and "Probability" keys exactly as shown:
"Top_1", "Top_2", "Top_3": The predicted exact scores and their probabilities.
Every exact Scoreline must strictly follow the order of NameofTeam1 - NameofTeam2.
This is an example with Team1:Norway and Team2:France
{{
  "Poisson Reasoning": "Your analyses description",
  "Top_1": {{
    "Score": "Norway 0-1 France",
    "Probability": "12.49%"
  }},
  "Top_2": {{
    "Score": "Norway 1-1 France",
    "Probability": "11.87%"
  }},
  "Top_3": {{
    "Score": "Norway 0-2 France",
    "Probability": "9.99%"
  }}
}}
'''

agent_a_response = ask_agent_with_search(prompt_agent_a)
#print(agent_a_response)



In [604]:
print(prompt_agent_a)

Within the next hours we have the fifa worldcup game between Brazil vs Japan. 
You are a quantitative football data scientist analyzing this game in order to predict the Top-3 probabilities of exact outcomes. 
CRITICAL RULE (Zero Data Leakage): You must absolutely ignore all betting odds, implied probabilities, and bookmaker stats. When using your web search tool, restrict your queries to raw statistical databases (e.g., FBref, Transfermarkt, official FIFA stats). NEVER search for "predictions", "odds", or "betting tips". 
Chain-of-Thought Guardrail: Football is more or less a low-scoring game well approximated by independent Poisson distributions. 
Before outputting any probabilities, mathematically reason step-by-step about the base expected goals (lambda_Team1 and lambda_Team2) for this specific match. 
Variables & Weighting: Base your lambda estimation on these factors: Heavily weight recent match results (last 5-10 games) and short-term xG trends. 
Further factor in: overall attac

In [ ]:
#Example Agent A (Remove cell if not needed)
agent_a_response = '''
{
"Poisson Reasoning": "To estimate the match probabilities on neutral ground in Houston, I calculated the base expected goals using recent xG trends, historical head-to-head records, and overall squad metrics like market value and Elo ratings. This yielded an expected goal rate (lambda) of 1.6 for Brazil and 0.8 for Japan, reflecting Brazil's superior attacking strength. Applying an independent Poisson distribution to these lambdas generates the most likely exact scorelines. The calculations directly point to a low-scoring Brazilian victory or a narrow draw as the most probable outcomes.",
"Top_1": {
"Score": "Brazil 1-0 Japan",
"Probability": "14.50%"
},
"Top_2": {
"Score": "Brazil 1-1 Japan",
"Probability": "11.60%"
},
"Top_3": {
"Score": "Brazil 2-0 Japan",
"Probability": "11.58%"
}
}
'''

In [ ]:
#Agent B
prompt_agent_b= intro + " "+f'''
You are an news-based investigative football scout analyzing this game in order to predict the Top-3 probabilities of exact outcomes. Base your predictions only on regular time (+ stoppage time), excluding extra time and penalty shootouts.
CRITICAL RULE (Zero Data Leakage):Use web search for the latest news (last 48h) on the teams. You must absolutely ignore all betting odds, implied probabilities, and bookmaker stats. Use web search specifically to find the latest news (last 48h), press conferences, and injury reports. You must absolutely ignore all betting odds, implied probabilities, and bookmaker stats. Actively filter out any market sentiment from the articles you read.
Chain-of-Thought Guardrail: Football is more or less a low-scoring game well approximated by independent Poisson distributions. Before outputting any probabilities, mathematically reason step-by-step about how current intelligence shifts 
the expected goals (lambda_Team1 and lambda_Team2) from a statistical baseline. 
Variables: Base your lambda shifts strictly on qualitative conditions: player availability/injuries (did the trainer or team officials announece any last minutes changes?), recent and emerging form of the team, tactical matchups, pressing intensity, set-piece efficiency, climate adaptation (heat/humidity/altitude), squad depth, and big-match experience, any other relevant news for this match. 
Include "Host Nation Crowd Psychology" only if USA, Mexico, or Canada is playing. 
Your Output should be strictly & only JSON: "Narrative_to_Poisson_Mapping": Step-by-step reasoning on how the retrieved intelligence specifically alters lambda_Team1 and lambda_Team2 without inflating overall goal expectations (max 5 sentences). 
Your Output must be strictly and ONLY a valid JSON object matching the exact structure and keys below. Do not format the Top 3 predictions as flat strings; you must use nested objects with "Score" and "Probability" keys exactly as shown:
"Top_1", "Top_2", "Top_3": The predicted exact scores and their probabilities.
Every exact Scoreline must strictly follow the order of NameofTeam1 - NameofTeam2.
This is an example with Team1:Norway and Team2:France
{{
  "Narrative_to_Poisson_Mapping": "Your analyses description",
  "Top_1": {{
    "Score": "Norway 0-1 France",
    "Probability": "12.49%"
  }},
  "Top_2": {{
    "Score": "Norway 1-1 France",
    "Probability": "11.87%"
  }},
  "Top_3": {{
    "Score": "Norway 0-2 France",
    "Probability": "9.99%"
  }}
}}
'''
agent_b_response = ask_agent_with_search(prompt_agent_b)
#print(agent_b_response)

In [607]:
print(prompt_agent_b)

Within the next hours we have the fifa worldcup game between Brazil vs Japan. 
You are an news-based investigative football scout analyzing this game in order to predict the Top-3 probabilities of exact outcomes. 
CRITICAL RULE (Zero Data Leakage):Use web search for the latest news (last 48h) on the teams. You must absolutely ignore all betting odds, implied probabilities, and bookmaker stats. Use web search specifically to find the latest news (last 48h), press conferences, and injury reports. You must absolutely ignore all betting odds, implied probabilities, and bookmaker stats. Actively filter out any market sentiment from the articles you read.
Chain-of-Thought Guardrail: Football is more or less a low-scoring game well approximated by independent Poisson distributions. Before outputting any probabilities, mathematically reason step-by-step about how current intelligence shifts 
the expected goals (lambda_Team1 and lambda_Team2) from a statistical baseline. 
Variables: Base your l

In [ ]:
#Example Agent B (Remove cell if not needed)
agent_b_response = '''
{
"Narrative_to_Poisson_Mapping": "Brazil enters the Round of 32 in formidable form, highlighted by consecutive 3-0 victories and Vinicius Jr.'s four goals, which significantly shifts their baseline lambda upwards. Japan's resilient attack, featuring in-form Kamada and Ueda and a consistent group stage scoring record, elevates their offensive expected goals. However, Brazil's sturdy defense, anchored by Marquinhos and Gabriel Magalhães, slightly tempers Japan's expected output. The high-stakes knockout environment at Houston Stadium typically compresses overall goal expectations due to tactical caution. Balancing Brazil's potent, high-intensity attack against Japan's robust counter-threat yields a higher statistical probability for a narrow Brazilian victory or a competitive draw.",
"Top_1": {
"Score": "Brazil 2-1 Japan",
"Probability": "12.45%"
},
"Top_2": {
"Score": "Brazil 1-1 Japan",
"Probability": "10.85%"
},
"Top_3": {
"Score": "Brazil 2-0 Japan",
"Probability": "9.75%"
}
}
'''

In [ ]:
#Agent C, no probabilities necessary because no structured probabilities are required, raw text containing thoughts is sufficient. 

prompt_agent_c= intro + " "+f'''
You are the Skeptic. Evaluate the independent analyses: Quant [Input A from Agent A: {agent_a_response}] and Qual [Input B from Agent B: {agent_b_response}]. 
CRITICAL RULE (Zero Data Leakage): Under no circumstances use betting markets or consensus favorites to validate or invalidate A and B. When using your web search, use it STRICTLY to fact-check the specific claims made in Input A and B (e.g., verifying if a player is actually injured, or if an xG stat is accurate). Do not search for general match predictions. 
Chain-of-Thought Guardrail: You try to find potential weaknesses of Input A (Agent A) and Input B (Agent B) provided. 
Your Output: Return strictly as a JSON object with: "Critique_Reasoning": 
Step-by-step logical deconstruction of the lambda assumptions in both models (max 5 sentences). 
"Critique_Quant": A concise, targeted critique of Agent A's variables (focusing on recent form vs. historical data). "Critique_Qual": 
A concise, targeted critique of Agent B's variables. 
'''
agent_c_response = ask_agent_with_search(prompt_agent_c)
#print(agent_c_response)

In [610]:
print(prompt_agent_c)

Within the next hours we have the fifa worldcup game between Brazil vs Japan. 
You are the Skeptic. Evaluate the independent analyses: Quant [Input A from Agent A: 
{
"Poisson Reasoning": "To estimate the match probabilities on neutral ground in Houston, I calculated the base expected goals using recent xG trends, historical head-to-head records, and overall squad metrics like market value and Elo ratings. This yielded an expected goal rate (lambda) of 1.6 for Brazil and 0.8 for Japan, reflecting Brazil's superior attacking strength. Applying an independent Poisson distribution to these lambdas generates the most likely exact scorelines. The calculations directly point to a low-scoring Brazilian victory or a narrow draw as the most probable outcomes.",
"Top_1": {
"Score": "Brazil 1-0 Japan",
"Probability": "14.50%"
},
"Top_2": {
"Score": "Brazil 1-1 Japan",
"Probability": "11.60%"
},
"Top_3": {
"Score": "Brazil 2-0 Japan",
"Probability": "11.58%"
}
}
] and Qual [Input B from Agent B: 


In [ ]:
#Example Agent C (Remove cell if not needed)
agent_c_response = '''
{
"Critique_Reasoning": "Both models attempt to adjust base scoring rates (lambdas) for the Round of 32 matchup, but they anchor their assumptions on fundamentally divergent data types. Agent A's lambdas are artificially suppressed by relying on long-term historical head-to-head data and static Elo ratings, which dilute the impact of current tournament momentum. Conversely, Agent B arbitrarily inflates expected goals by overweighting recent narrative peaks—such as Vinicius Jr.'s four group-stage goals and Japan's rout of Tunisia—without applying rigorous regression to the mean. Neither model adequately adjusts for the stylistic clash of Brazil's high possession against Japan's lethal transition game. Ultimately, deriving exact decimal probabilities from these heavily skewed base lambdas creates a false sense of certainty in both the quantitative and qualitative outputs.",
"Critique_Quant": "Agent A's reliance on historical head-to-head records, overall market values, and Elo ratings creates a static model that ignores current tournament form. By weighting these backward-looking variables so heavily, Agent A suppresses goal expectations and fails to account for the immediate, real-time impact of in-form squads, treating a dynamic 2026 World Cup knockout match as a generic historical average.",
"Critique_Qual": "Agent B over-indexes on subjective group stage narratives, using performances against much weaker opposition (like Brazil's win over Scotland and Japan's win over Tunisia) to justify shifting baseline lambdas. This qualitative approach artificially inflates attacking metrics based on momentum and 'knockout environment' tropes, lacking the mathematical rigidity needed to justify its hyper-specific fractional probabilities."
}
'''


In [ ]:
# Agent E
prompt_agent_e= intro + " "+f'''
You are the Meta-Analyst and Final Adjudicator analyzing this game in order to predict the Top-3 probabilities of exact outcomes. Base your predictions only on regular time (+ stoppage time), excluding extra time and penalty shootouts.
Synthesize the independent analyses from the Agent A [{agent_a_response}], the Agent B [{agent_b_response}], and the Agent C [{agent_c_response}] 
for the FIFA World Cup match. 
CRITICAL RULE (Zero Data Leakage & Search Directive): Under no circumstances use betting markets, odds, or consensus favorites to calibrate your final prediction. 
If you use your web search tool, use it STRICTLY to resolve factual deadlocks (e.g., checking if a player is actually injured after conflicting reports).
NEVER search for "match predictions", "betting odds", or "consensus picks". Actively filter out any market sentiment from search results. 
Anti-Averaging & True Adjudication Guardrail: Do NOT simply average the lambdas or probabilities of Agents A and B. You must critically evaluate ALL three inputs.
REMEMBER: Agent C (The Skeptic) is not infallible and can also be wrong. You can also argue any assumptions of Agents A and B (e.g. the low scoring poisson-like distribution)
If Agent C's critique is logically flawed, overly dismissive, or ignores strong data, you must boldly overrule Agent C and defend the valid insights from A or B. 
If Agent C correctly identifies flaws, adjust accordingly. If all three agents (A, B, and C) are flawed, independently formulate a new, accurate lambda recalibration. 
Chain-of-Thought Guardrail: Use simulated Bayesian updating to form a final probability distribution. Your main goal is to predict the highest likely outcomes.
Your Output should be strictly & only JSON: Return strictly as a JSON object with: "Bayesian_CoT": Step-by-step logical synthesis explaining how the final lambdas were calibrated, explicitly stating whose logic (including C's) you validated, overruled, or if you had to independently recalibrate (max 5 sentences). "Top_1", "Top_2", "Top_3": The final calibrated exact scores and their probabilities.
Your Output must be strictly and ONLY a valid JSON object matching the exact structure and keys below. Do not format the Top 3 predictions as flat strings; you must use nested objects with "Score" and "Probability" keys exactly as shown:
"Top_1", "Top_2", "Top_3": The predicted exact scores and their probabilities.
Every exact Scoreline must strictly follow the order of NameofTeam1 - NameofTeam2.
This is an example with Team1:Norway and Team2:France
{{
  "Bayesian_CoT": "Your analyses description",
  "Top_1": {{
    "Score": "Norway 0-1 France",
    "Probability": "12.49%"
  }},
  "Top_2": {{
    "Score": "Norway 1-1 France",
    "Probability": "11.87%"
  }},
  "Top_3": {{
    "Score": "Norway 0-2 France",
    "Probability": "9.99%"
  }}
}}
'''
agent_e_response = ask_agent_with_search(prompt_agent_e)
#print(agent_e_response)

In [613]:
print(prompt_agent_e)

Within the next hours we have the fifa worldcup game between Brazil vs Japan. 
You are the Meta-Analyst and Final Adjudicator analyzing this game in order to predict the Top-3 probabilities of exact outcomes. 
Synthesize the independent analyses from the Agent A [
{
"Poisson Reasoning": "To estimate the match probabilities on neutral ground in Houston, I calculated the base expected goals using recent xG trends, historical head-to-head records, and overall squad metrics like market value and Elo ratings. This yielded an expected goal rate (lambda) of 1.6 for Brazil and 0.8 for Japan, reflecting Brazil's superior attacking strength. Applying an independent Poisson distribution to these lambdas generates the most likely exact scorelines. The calculations directly point to a low-scoring Brazilian victory or a narrow draw as the most probable outcomes.",
"Top_1": {
"Score": "Brazil 1-0 Japan",
"Probability": "14.50%"
},
"Top_2": {
"Score": "Brazil 1-1 Japan",
"Probability": "11.60%"
},
"To

In [ ]:
#Example Agent E (Remove cell if not needed)
agent_e_response = '''
{
"Bayesian_CoT": "We accept Agent C's valid critique that Agent A relies too heavily on static historical data and Agent B over-indexes on unregressed group-stage anomalies. However, we overrule Agent C's complete dismissal of recent momentum, recognizing that short-tournament knockout form is highly relevant when tempered by tactical caution. Synthesizing these perspectives, we recalibrate the lambdas to account for the specific stylistic clash of Brazil's high possession against Japan's lethal transition threat. This mathematical adjustment yields an optimized scoring distribution that avoids both Agent A's underestimation and Agent B's inflation of goal rates. The resulting Poisson matrix favors a narrow, competitive Brazilian advantage or a tight draw as the most probable exact outcomes.",
"Top_1": {
"Score": "Brazil 1-0 Japan",
"Probability": "13.61%"
},
"Top_2": {
"Score": "Brazil 1-1 Japan",
"Probability": "12.25%"
},
"Top_3": {
"Score": "Brazil 2-0 Japan",
"Probability": "10.21%"
}
}
'''

In [615]:
#Central Ledger, change to temp for initial check?
import json
from datetime import datetime, timezone

def safe_parse_json(agent_name: str, raw_string: str) -> dict:
    """Helper method to safely parse LLM output into a dictionary."""
    try:
        # Strip potential markdown formatting that LLMs sometimes add (e.g., ```json ... ```)
        clean_string = raw_string.strip()
        if clean_string.startswith("```json"):
            clean_string = clean_string[7:-3].strip()
            
        return json.loads(clean_string)
    except json.JSONDecodeError:
        print(f"Warning: Failed to parse JSON for {agent_name}. Saving as raw text instead.")
        return {"Only Raw_Text Available": raw_string}


def log_full_match_to_ledger(match_id: str, outputs: dict, bookmaker_data: dict, ledger_path: str = "predictions_ledger.jsonl"): #CHANGE HERE?
    """
    Saves the complete state of the DAG (All Agents + Bookmaker) into a single ledger entry.
    """
    
    # 1. Safely parse all agent outputs
    parsed_agents = {
        "Agent_A_Quant": safe_parse_json("Agent_A", outputs.get('agent_a', '{}')),
        "Agent_B_Qual": safe_parse_json("Agent_B", outputs.get('agent_b', '{}')),
        "Agent_C_Critique": safe_parse_json("Agent_C", outputs.get('agent_c', '{}')),
        "Agent_E_Meta": safe_parse_json("Agent_E", outputs.get('agent_e', '{}'))
    }

    # 2. Build the master record
    master_record = {
        "metadata": {
            "match_id": match_id,
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        },
        "bookmaker_baseline": bookmaker_data,
        "dag_pipeline": parsed_agents
    }
    
    # 3. Append to the JSONL file
    with open(ledger_path, 'a', encoding='utf-8') as f:
        f.write(json.dumps(master_record) + '\n')
        
    print(f"Successfully logged full DAG prediction for {match_id} to {ledger_path}")

# ==========================================
# Example Usage in your Orchestrator (main.py):
# ==========================================

# 1. Run your agents (Simulated strings here)
out_a = agent_a_response
out_b = agent_b_response
out_c = agent_c_response
out_e = agent_e_response



# 2. Pack agent outputs into a dictionary
agent_results = {
    "agent_a": out_a,
    "agent_b": out_b,
    "agent_c": out_c,
    "agent_e": out_e
}
    
# 3. Log everything at once!
log_full_match_to_ledger(
    match_id=f"{Team1}-{Team2}", 
    outputs=agent_results, 
    bookmaker_data=bookmaker_odds
)

Successfully logged full DAG prediction for Brazil-Japan to predictions_ledger.jsonl


In [616]:
#Git commands
!git status

On branch update-files-branch
Your branch is up to date with 'origin/update-files-branch'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   Main-Analysis.ipynb
	modified:   predictions_ledger.jsonl

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.env
	cleaned_predictions.csv
	evaluation_ledger.ipynb
	predictions_ledgertemp.jsonl

no changes added to commit (use "git add" and/or "git commit -a")


In [617]:
!git add predictions_ledger.jsonl

In [618]:
!git commit -m "updated ({Team1}-{Team2}) in predictions_ledger.jsonl"
#!git commit -m "doublette-prediction removed Panama-England in predictions_ledger.jsonl"


[update-files-branch e4bffab] updated (Brazil-Japan) in predictions_ledger.jsonl
 1 file changed, 1 insertion(+)


In [619]:
!git push -u origin update-files-branch

Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 10 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 2.23 KiB | 2.23 MiB/s, done.
Total 3 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/VargheseLab/Fifa2026-Pred.git
   ddc4b7f..e4bffab  update-files-branch -> update-files-branch
branch 'update-files-branch' set up to track 'origin/update-files-branch'.
